### 1. Tải bộ dữ liệu:

In [1]:
# !gdown --id 1qiUDDoYyRLBiKOoYWdFl_5WByHE8Cugu

### 2. Import các thư viện cần thiết:

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

### 3. Cài đặt giá trị ngẫu nhiên cố định:

In [2]:
random_state = 59
np.random.seed(random_state)
torch.manual_seed(random_state)
if torch.cuda.is_available():
	torch.cuda.manual_seed(random_state)

### 4. Cài đặt thiết bị tính toán:

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

### 5. Đọc bộ dữ liệu:

In [4]:
dataset_path = 'Auto_MPG_data.csv'
dataset = pd.read_csv(dataset_path)

### 6. Tiền xử lý bộ dữ liệu:

#### (a) Tách đặc trưng X và nhãn y:

In [5]:
X = dataset.drop(columns = 'MPG').values
y = dataset['MPG'].values

#### (b) Chia bộ dữ liệu train/val/test:

In [6]:
val_size = 0.2
test_size = 0.125
is_shuffle = True

X_train, X_val, y_train, y_val = train_test_split(
	X, y,
	test_size = val_size,
	random_state = random_state,
	shuffle = is_shuffle
)

X_train, X_test, y_train, y_test = train_test_split(
	X_train, y_train,
	test_size = test_size,
	random_state = random_state,
	shuffle = is_shuffle
)

#### (c) Chuẩn hóa đặc trưng đầu vào:

In [7]:
normalizer = StandardScaler()
X_train = normalizer.fit_transform(X_train)
X_val = normalizer.transform(X_val)
X_test = normalizer.transform(X_test)

X_train = torch.tensor(X_train, dtype = torch.float32)
X_val = torch.tensor(X_val, dtype = torch.float32)
X_test = torch.tensor(X_test, dtype = torch.float32)
y_train = torch.tensor(y_train, dtype = torch.float32)
y_val = torch.tensor(y_val, dtype = torch.float32)
y_test = torch.tensor(y_test, dtype = torch.float32)

### 7. Xây dựng DataLoader:

In [8]:
class CustomDataset(Dataset):
	def __init__(self, X, y):
		self.X = X
		self.y = y

	def __len__(self):
		return len(self.y)

	def __getitem__(self, idx):
		return self.X[idx], self.y[idx]

In [9]:
train_dataset = CustomDataset(X_train, y_train)
val_dataset = CustomDataset(X_val, y_val)
test_dataset = CustomDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size = 64, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = 64, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = 64, shuffle = True)

### 8. Xây dựng mạng MLP:

In [67]:
class MLP(nn.Module):
	def __init__(self, input_dims, hidden_dims, output_dims):
		super().__init__()
		self.linear1 = nn.Linear(input_dims, hidden_dims)
		self.linear2 = nn.Linear(hidden_dims, hidden_dims)
		self.output = nn.Linear(hidden_dims, output_dims)

	def forward(self, x):
		x = self.linear1(x)
		# x = F.relu(x)
		# x = torch.sigmoid(x)
		x = F.tanh(x)
		x = self.linear2(x)
		# x = F.relu(x)
		# x = F.sigmoid(x)
		x = F.tanh(x)
		out = self.output(x)
		return out.squeeze(1)


In [55]:
class LinearRegressionModel(nn.Module):
	def __init__(self, input_dims, output_dims):
		super().__init__()
		self.output = nn.Linear(input_dims, output_dims)

	def forward(self, x):
		y_pred = self.output(x)
		return y_pred.squeeze(1)

In [62]:
input_dims = X_train.shape[1]
output_dims = 1
hidden_dims = 64

model = MLP(input_dims = input_dims,
            hidden_dims = hidden_dims,
            output_dims = output_dims).to(device)

#
# model = LinearRegressionModel(input_dims = input_dims,
#             output_dims = output_dims).to(device)

### 9. Khai báo hàm loss và optimizer:

In [63]:
lr = 1e-2
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr = lr)

### 10. Xây dựng hàm tính điểm R2:

In [64]:
def r_squared(y_true, y_pred):
	y_true = torch.Tensor(y_true).to(device)
	y_pred = torch.Tensor(y_pred).to(device)
	mean_true = torch.mean(y_true)
	ss_tot = torch.sum((y_true - mean_true) ** 2)
	ss_res = torch.sum((y_true - y_pred) ** 2)
	r2 = 1 - (ss_res / ss_tot)
	return r2

### 11. Huấn luyện mô hình:

In [65]:
epochs = 100
train_losses = []
val_losses = []
train_r2 = []
val_r2 = []

for epoch in range(epochs):
	train_loss = 0.0
	train_target = []
	val_target = []
	train_predict = []
	val_predict = []
	model.train()
	for X_samples, y_samples in train_loader:
		X_samples = X_samples.to(device)
		y_samples = y_samples.to(device)
		optimizer.zero_grad()
		outputs = model(X_samples)
		train_predict += outputs.tolist()
		train_target += y_samples.tolist()
		loss = criterion(outputs, y_samples)
		loss.backward()
		optimizer.step()
		train_loss += loss.item()
	train_loss /= len(train_loader)
	train_losses.append(train_loss)
	train_r2.append(r_squared(train_target, train_predict))
	model.eval()
	val_loss = 0.0
	with torch.no_grad():
		for X_samples, y_samples in val_loader:
			X_samples = X_samples.to(device)
			y_samples = y_samples.to(device)
			outputs = model(X_samples)
			val_predict += outputs.tolist()
			val_target += y_samples.tolist()
			loss = criterion(outputs, y_samples)
			val_loss += loss.item()
	val_loss /= len(val_loader)
	val_losses.append(val_loss)
	val_r2.append(r_squared(val_target, val_predict))
	print(f'\nEPOCH {epoch + 1}:\tTraining loss : {train_loss:.3f}\tValidation loss : {val_loss:.3f}')


EPOCH 1:	Training loss : 381.836	Validation loss : 47.513

EPOCH 2:	Training loss : 43.075	Validation loss : 14.633

EPOCH 3:	Training loss : 20.610	Validation loss : 12.453

EPOCH 4:	Training loss : 18.760	Validation loss : 15.979

EPOCH 5:	Training loss : 17.674	Validation loss : 7.640

EPOCH 6:	Training loss : 15.713	Validation loss : 13.366

EPOCH 7:	Training loss : 16.139	Validation loss : 11.020

EPOCH 8:	Training loss : 13.977	Validation loss : 8.925

EPOCH 9:	Training loss : 13.251	Validation loss : 7.138

EPOCH 10:	Training loss : 11.587	Validation loss : 7.716

EPOCH 11:	Training loss : 10.210	Validation loss : 8.051

EPOCH 12:	Training loss : 9.754	Validation loss : 8.535

EPOCH 13:	Training loss : 9.964	Validation loss : 6.186

EPOCH 14:	Training loss : 9.503	Validation loss : 5.870

EPOCH 15:	Training loss : 10.790	Validation loss : 6.187

EPOCH 16:	Training loss : 8.365	Validation loss : 5.914

EPOCH 17:	Training loss : 9.517	Validation loss : 4.936

EPOCH 18:	Training l

### 12. Đánh giá mô hình:

In [66]:
model.eval()
with torch.no_grad():
	y_hat = model(X_test)
	test_set_r2 = r_squared(y_hat, y_test)
	print('Evaluation on test set :')
	print(f'R2: {test_set_r2}')

Evaluation on test set :
R2: 0.8968321084976196
